# long-term memory with SQLite

In [ ]:
%load_ext dotenv
%dotenv
%load_ext mypy_ipython

# import relevant classes and functions

In [ ]:
from langchain.graph import START , END , MessageState
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage , SystemMessage , RemoveMessage , AIMessage
from langgraph.checkpoint.memory import InMemorySaver
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

In [ ]:
class State(MessageState):
    summary: str

# Define the nodes¶

In [ ]:
chat = ChatOpenAI(model = 'gpt-4o',
                 seed = 365,
                 temperature = 0,
                 max_completion_tokens = 100)

In [ ]:
def ask_questions(state: State) -> State:
    print(f'\n------> ENTERRING ask_question:')
    for i in state['messages']:
        i.preety_print()

    question = "what is your question"
    print(question)
    
    return State(messages = [AIMessage(question) , HumanMessage(input())])

In [ ]:
f chatbot(state: state) -> State:
    print(f'\n------> ENTERING ask_another_question:')
    for i in state['message']:
        i.pretty_print()

    system_message = f'''
    here a quick summary of what been discussed so far:
    {state.get("summary" , "")}

    keep this in mind as your answer the next question.
    '''
        
    response = chat.invoke([SystemMessage(system_message)] + state["message"])
    response.pretty_print()

    return State(messages = [response])

In [ ]:
def ask_another_questions(state: State) -> State:
    print(f'\n------> ENTERING ask_another_question:')

    question = "Would you like to ask one more question (yes/no)?")
    print(question)
    return State(messages = [HumanMessage(input())])

In [ ]:
def summarize_and_delete_message(state: MessagesState) -> MessagesState:
    print(f'\n------> ENTERING ask_another_question:')
    new_conversation = ""
    for i in state["message"]:
        
        new_conversation += f"{i.type}: {i.content}\n\n"

    summary_instructions = f'''
update the ongoing summary by incorporating the new lines of conversation below.

previous summary:
{state.get("summary" , "")}

New conversation:
{new_conveersation}
''' 

    print(summary_insrtructions)
    summary = chat.invoke([HumanMessage(summary_instructions)])

    
    remove_messages = [RemoveMessage(id = i.id) for i in my_list[:-5]]

    return MessagesState(messages = remove_message , summary = summary.content)

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node("ask_question" , ask_question)
graph.add_node("chatbot" , chatbot)
graph.add_node("ask_another_question" , ask_another_question)

graph.add_edge("START" , "ask_question")
graph.add_edge("ask_question" ,"chatbot")
graph.add_edge("chatbot" , "ask_another_question")
graph.add_conditional_edges(source = 'ask_another_question' , path = routing_function, path_map = {"True":"ask_question" ,'false': "__end__"})


In [ ]:
db_path = "C:/users/hristina/desktop/langgraph_DB/langgraph.db"
con = sqlite3.connect(database = db_path , check_thread = False)

In [ ]:
checkpointer = SqliteSaver(con)

In [ ]:
graph_compiled = graph.compile(checkpointer)

In [ ]:
graph_compiled

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
config2 = {"configurable": {"thread_id": "2"}}


In [ ]:
graph_compiled.invoke(State() , config2)